In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_resumo_cliente_mes
USING DELTA
AS

WITH movimentacoes AS (

    SELECT
        id_cliente,
        nome_cliente,
        DATE_TRUNC('MONTH', data_transacao)
            AS mes_referencia,
        id_transacao,
        valor,
        valor_liquido,
        estornada

    FROM workspace.desafio_sicredi.gold_transacoes_cartao

),

resumo_mensal AS (

    SELECT
        id_cliente,
        nome_cliente,
        mes_referencia,

        COUNT(*) AS total_transacoes,

        SUM(
            CASE
                WHEN estornada = FALSE THEN 1
                ELSE 0
            END
        ) AS transacoes_nao_estornadas,

        SUM(
            CASE
                WHEN estornada = TRUE THEN 1
                ELSE 0
            END
        ) AS total_estornos,

        SUM(valor) AS valor_bruto,

        SUM(valor_liquido) AS valor_liquido,

        AVG(
            CASE
                WHEN estornada = FALSE
                THEN valor_liquido
            END
        ) AS ticket_medio

    FROM movimentacoes

    GROUP BY
        id_cliente,
        nome_cliente,
        mes_referencia
)

SELECT *
FROM resumo_mensal;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    id_cliente,
    nome_cliente,
    mes_referencia,
    total_transacoes,
    total_estornos,
    valor_bruto,
    valor_liquido,
    ticket_medio

FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

ORDER BY id_cliente, mes_referencia;

id_cliente,nome_cliente,mes_referencia,total_transacoes,total_estornos,valor_bruto,valor_liquido,ticket_medio
1,Ana Maria Silva,2026-10-01T00:00:00.000Z,4,1,880.00,800.00,266.666667
3,Mariana Costa,2026-10-01T00:00:00.000Z,1,0,300.00,300.00,300.000000


In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes_mes,
    SUM(total_transacoes) AS total_transacoes,
    SUM(total_estornos) AS total_estornos,
    SUM(valor_bruto) AS valor_bruto,
    SUM(valor_liquido) AS valor_liquido

FROM workspace.desafio_sicredi.gold_resumo_cliente_mes;

total_clientes_mes,total_transacoes,total_estornos,valor_bruto,valor_liquido
2,5,1,1180.00,1100.00


In [0]:
%sql

WITH evolucao AS (

    SELECT
        id_cliente,
        nome_cliente,
        mes_referencia,
        valor_liquido,

        LAG(valor_liquido) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS valor_mes_anterior,

        LAG(mes_referencia) OVER (
            PARTITION BY id_cliente
            ORDER BY mes_referencia
        ) AS mes_anterior

    FROM workspace.desafio_sicredi.gold_resumo_cliente_mes

)

SELECT
    id_cliente,
    nome_cliente,
    mes_referencia,
    valor_liquido,
    valor_mes_anterior,

    ROUND(
        (
            (valor_liquido - valor_mes_anterior)
            / NULLIF(valor_mes_anterior, 0)
        ) * 100,
        2
    ) AS crescimento_percentual

FROM evolucao

WHERE mes_anterior = ADD_MONTHS(mes_referencia, -1)
  AND valor_mes_anterior > 0
  AND valor_liquido > valor_mes_anterior * 1.5;

id_cliente,nome_cliente,mes_referencia,valor_liquido,valor_mes_anterior,crescimento_percentual


In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_ranking_estabelecimento
USING DELTA
AS

WITH transacoes_validas AS (

    SELECT
        id_transacao,
        nome_estabelecimento,
        cidade_estabelecimento,
        valor_liquido

    FROM workspace.desafio_sicredi.gold_transacoes_cartao

    WHERE estornada = FALSE
      AND valor_liquido > 0

),

resumo_estabelecimento AS (

    SELECT
        nome_estabelecimento,
        cidade_estabelecimento,

        COUNT(DISTINCT id_transacao)
            AS quantidade_transacoes,

        SUM(valor_liquido)
            AS valor_total,

        AVG(valor_liquido)
            AS ticket_medio

    FROM transacoes_validas

    GROUP BY
        nome_estabelecimento,
        cidade_estabelecimento

),

ranking AS (

    SELECT
        *,

        DENSE_RANK() OVER (
            PARTITION BY cidade_estabelecimento
            ORDER BY valor_total DESC
        ) AS ranking_valor,

        DENSE_RANK() OVER (
            PARTITION BY cidade_estabelecimento
            ORDER BY quantidade_transacoes DESC
        ) AS ranking_quantidade

    FROM resumo_estabelecimento

)

SELECT *
FROM ranking;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    cidade_estabelecimento,
    nome_estabelecimento,
    quantidade_transacoes,
    valor_total,
    ticket_medio,
    ranking_valor,
    ranking_quantidade

FROM workspace.desafio_sicredi.gold_ranking_estabelecimento

ORDER BY
    cidade_estabelecimento,
    ranking_valor;

cidade_estabelecimento,nome_estabelecimento,quantidade_transacoes,valor_total,ticket_medio,ranking_valor,ranking_quantidade
PAULISTA,POSTO CENTRAL,1,300.00,300.000000,1,1
RECIFE,LOJA ATRASADA,1,450.00,450.000000,1,2
RECIFE,SUPERMERCADO RECIFE,2,350.00,175.000000,2,1


In [0]:
%sql

SELECT
    COUNT(*) AS total_estabelecimentos,
    SUM(quantidade_transacoes) AS total_transacoes,
    SUM(valor_total) AS valor_total_movimentado

FROM workspace.desafio_sicredi.gold_ranking_estabelecimento;

total_estabelecimentos,total_transacoes,valor_total_movimentado
3,4,1100.00


In [0]:
%sql

SELECT
    cidade_estabelecimento,
    nome_estabelecimento,
    valor_total,
    ranking_valor

FROM workspace.desafio_sicredi.gold_ranking_estabelecimento

WHERE ranking_valor = 1

ORDER BY cidade_estabelecimento;

cidade_estabelecimento,nome_estabelecimento,valor_total,ranking_valor
PAULISTA,POSTO CENTRAL,300.00,1
RECIFE,LOJA ATRASADA,450.00,1


In [0]:
%sql

CREATE OR REPLACE TABLE
workspace.desafio_sicredi.gold_alertas_qualidade
USING DELTA
AS

WITH registros_quarentena AS (

    SELECT
        'CLIENTES' AS entidade,
        motivo_erro
    FROM workspace.desafio_sicredi.quarentena_clientes

    UNION ALL

    SELECT
        'CONTAS' AS entidade,
        motivo_erro
    FROM workspace.desafio_sicredi.quarentena_contas

    UNION ALL

    SELECT
        'CARTOES' AS entidade,
        motivo_erro
    FROM workspace.desafio_sicredi.quarentena_cartoes

    UNION ALL

    SELECT
        'TRANSACOES_CARTAO' AS entidade,
        motivo_erro
    FROM workspace.desafio_sicredi.quarentena_transacoes_cartao

    UNION ALL

    SELECT
        'ESTORNOS' AS entidade,
        motivo_erro
    FROM workspace.desafio_sicredi.quarentena_estornos
),

resumo_qualidade AS (

    SELECT
        entidade,
        motivo_erro,
        COUNT(*) AS quantidade_registros

    FROM registros_quarentena

    GROUP BY
        entidade,
        motivo_erro
)

SELECT
    entidade,
    motivo_erro,
    quantidade_registros,
    CURRENT_DATE() AS data_processamento

FROM resumo_qualidade;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    entidade,
    motivo_erro,
    quantidade_registros,
    data_processamento

FROM workspace.desafio_sicredi.gold_alertas_qualidade

ORDER BY
    entidade,
    motivo_erro;

entidade,motivo_erro,quantidade_registros,data_processamento
CARTOES,CONTA_INEXISTENTE_OU_INVALIDA,2,2026-10-15
CLIENTES,CPF_INVALIDO,1,2026-10-15
CONTAS,CLIENTE_INEXISTENTE_OU_INVALIDO,2,2026-10-15
CONTAS,STATUS_CONTA_INVALIDO,1,2026-10-15
ESTORNOS,TRANSACAO_INEXISTENTE_OU_INVALIDA,1,2026-10-15
TRANSACOES_CARTAO,CARTAO_INEXISTENTE_OU_INVALIDO,1,2026-10-15
TRANSACOES_CARTAO,DATA_TRANSACAO_INVALIDA,1,2026-10-15
TRANSACOES_CARTAO,VALOR_INVALIDO,1,2026-10-15


In [0]:
%sql

SELECT
    entidade,
    SUM(quantidade_registros) AS total_rejeitados

FROM workspace.desafio_sicredi.gold_alertas_qualidade

GROUP BY entidade

ORDER BY entidade;

entidade,total_rejeitados
CARTOES,2
CLIENTES,1
CONTAS,3
ESTORNOS,1
TRANSACOES_CARTAO,3


In [0]:
%sql

SELECT
    SUM(quantidade_registros)
        AS total_registros_rejeitados,

    COUNT(DISTINCT entidade)
        AS entidades_com_problemas,

    COUNT(DISTINCT motivo_erro)
        AS tipos_diferentes_erros

FROM workspace.desafio_sicredi.gold_alertas_qualidade;

total_registros_rejeitados,entidades_com_problemas,tipos_diferentes_erros
10,5,8


In [0]:
%sql

SHOW TABLES
IN workspace.desafio_sicredi
LIKE 'gold_*';

database,tableName,isTemporary
desafio_sicredi,gold_alertas_qualidade,false
desafio_sicredi,gold_ranking_estabelecimento,false
desafio_sicredi,gold_resumo_cliente_mes,false
desafio_sicredi,gold_transacoes_cartao,false


gold_transacoes_cartao
gold_resumo_cliente_mes
gold_ranking_estabelecimento
gold_alertas_qualidade